In [1]:
import sys

sys.path.append("..")

from ml.emotion import load_emotion_pipeline
emotion_pipeline = load_emotion_pipeline()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [2]:
result = emotion_pipeline("I am extremely happy and excited about this project!")
print(result)

[{'label': 'joy', 'score': 0.9912799000740051}]


In [8]:
import importlib
import ml.emotion

importlib.reload(ml.emotion)

<module 'ml.emotion' from 'd:\\Jupyter\\social-pulse-ai\\notebooks\\..\\ml\\emotion.py'>

In [4]:
from ml.emotion import predict_emotion

print(predict_emotion(
    "I am extremely happy and excited!",
    emotion_pipeline
))

print(predict_emotion(
    "I am really frustrated and angry about this.",
    emotion_pipeline
))

print(predict_emotion(None, emotion_pipeline))

{'label': 'joy', 'score': 0.9837809801101685}
{'label': 'anger', 'score': 0.9802591800689697}
{'label': None, 'score': None, 'error': 'Empty text'}


In [7]:
test_text = [
    "I am extremely happy! Today is the best day ever!",
    "I am furious about what happened. This is unacceptable!",
    "I feel very sad and disappointed today.",
    "I'm really scared about what might happen tomorrow.",
    "Wow! I never expected that to happen!"
]

for text in test_text:
    result = predict_emotion(text, emotion_pipeline)

    print(f"Text: {text}")
    print(f"Emotion: {result['label']}")
    print(f"Confidence: {result['score']:.4f}")
    print("-" * 60)

Text: I am extremely happy! Today is the best day ever!
Emotion: joy
Confidence: 0.9779
------------------------------------------------------------
Text: I am furious about what happened. This is unacceptable!
Emotion: anger
Confidence: 0.9875
------------------------------------------------------------
Text: I feel very sad and disappointed today.
Emotion: sadness
Confidence: 0.9882
------------------------------------------------------------
Text: I'm really scared about what might happen tomorrow.
Emotion: fear
Confidence: 0.9930
------------------------------------------------------------
Text: Wow! I never expected that to happen!
Emotion: surprise
Confidence: 0.9739
------------------------------------------------------------


In [9]:
from ml.emotion import analyze_emotion

print(analyze_emotion(
    "I finally completed my machine learning project!",
    emotion_pipeline
))

print(analyze_emotion(None, emotion_pipeline))

{'emotion': 'joy', 'confidence': 0.8922244906425476, 'error': None}
{'emotion': None, 'confidence': None, 'error': 'Empty text'}


In [10]:
import pandas as pd

large_df = pd.read_csv("../data/sentiment_results.csv")

print(large_df.shape)

(100, 10)


In [11]:
large_df["emotion_predictions"] = large_df["full_text"].apply(predict_emotion, args=(emotion_pipeline,))

In [12]:
large_df["emotion"] = large_df["emotion_predictions"].apply(
    lambda x: x["label"]
)

large_df["emotion_score"] = large_df["emotion_predictions"].apply(
    lambda x: x["score"]
)

In [13]:
print(large_df["emotion"].value_counts(dropna=False))

print("\nEmotion percentages:")
print(
    large_df["emotion"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
)

emotion
neutral     89
fear         6
anger        2
joy          1
sadness      1
surprise     1
Name: count, dtype: int64

Emotion percentages:
emotion
neutral     89.0
fear         6.0
anger        2.0
joy          1.0
sadness      1.0
surprise     1.0
Name: proportion, dtype: float64


In [14]:
non_neutral = large_df[
    large_df["emotion"] != "neutral"
][
    ["title", "emotion", "emotion_score"]
].sort_values(
    by="emotion_score",
    ascending=False
)

print(non_neutral.to_string(index=False))

                                                                                                        title  emotion  emotion_score
                                                                     Causal Inference Real World Case Studies      joy       0.981155
                                                                 Love DS, but terrified of one requirement...     fear       0.980364
                                                                             How to handle cofound variables?     fear       0.825148
                                                    Should I make the move from data science to product owner     fear       0.778200
                                 Thinking more seriously about the design of exploratory studies, a manifesto     fear       0.685532
         Spent months building a no-code data prep/ML tool — curious if the core idea actually matters to you surprise       0.671955
                                                             I

In [15]:
check_emotions = large_df[
    large_df["emotion"] != "neutral"
].sort_values("emotion_score")

for _, row in check_emotions.head(5).iterrows():
    print("=" * 80)
    print("TITLE:", row["title"])
    print("EMOTION:", row["emotion"])
    print("CONFIDENCE:", round(row["emotion_score"], 3))
    print("\nFULL TEXT:")
    print(row["full_text"][:1000])
    print()

TITLE: Seeking advice on the next research direction for an emergency landing site segmentation project
EMOTION: fear
CONFIDENCE: 0.432

FULL TEXT:
Seeking advice on the next research direction for an emergency landing site segmentation project [removed]

TITLE: Resigned after 1 month due to role mismatch + brutal commute — now company is holding my salary over a "bond"
EMOTION: anger
CONFIDENCE: 0.492

FULL TEXT:
Resigned after 1 month due to role mismatch + brutal commute — now company is holding my salary over a "bond"

TITLE: Data Science Online course
EMOTION: sadness
CONFIDENCE: 0.498

FULL TEXT:
Data Science Online course [removed]

TITLE: Chat bot frustrations
EMOTION: anger
CONFIDENCE: 0.505

FULL TEXT:
Chat bot frustrations [removed]

TITLE: Is AI more dangerous than the Nuclear Arms Race?
EMOTION: fear
CONFIDENCE: 0.65

FULL TEXT:
Is AI more dangerous than the Nuclear Arms Race?



In [16]:
large_df = large_df.drop(columns=["emotion_predictions"])

In [17]:
large_df.to_csv("../data/emotion_results.csv", index=False)

print("Saved successfully!")
print("Final dataset shape:", large_df.shape)
print("\nColumns:")
print(large_df.columns.tolist())

Saved successfully!
Final dataset shape: (100, 12)

Columns:
['created_utc', 'id', 'num_comments', 'score', 'selftext', 'subreddit', 'title', 'full_text', 'sentiment', 'sentiment_score', 'emotion', 'emotion_score']
